In [4]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
from models import ChurnLSTM, ChurnCNN1D

# ==========================================
# 1. 디렉토리 설정
# ==========================================
DATA_DIR = "./data/data/processed"
MODEL_DIR = "./model/deep_learning"
TEST_DIR = "./test"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("=" * 50)
print("🚀 [DL] PyTorch 데이터 로드 시작")
print("=" * 50)

# ==========================================
# 2. 시퀀스 데이터 및 타겟 로드
# ==========================================
seq_path = os.path.join(DATA_DIR, "converted_sequence_tensor.npy")
if not os.path.exists(seq_path):
    seq_path = os.path.join(DATA_DIR, "dl_sequence_tensor.npy")

X_seq = np.load(seq_path)

# 타겟 라벨 불러오기 (CSV에서 타겟 추출)
csv_path = os.path.join(DATA_DIR, "customer_features_20181101_v1.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(DATA_DIR, "train_20181101_v1.csv")

df_ml = pd.read_csv(csv_path)
target_col = "churn" if "churn" in df_ml.columns else ("is_churn" if "is_churn" in df_ml.columns else df_ml.columns[-1])
y = df_ml[target_col].values

# ==========================================
# 3. Data Split & DataLoader 구축
# ==========================================
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

X_seq_train, X_seq_test = X_seq[train_idx], X_seq[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

X_tr, X_val, y_tr, y_val = train_test_split(X_seq_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

BATCH_SIZE = 64
train_dataset = TensorDataset(torch.tensor(X_tr, dtype=torch.long), torch.tensor(y_tr, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.long), torch.tensor(y_val, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_seq_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.float32))

# 클래스 불균형 해결을 위한 Sampler 설정
class_counts = np.bincount(y_tr.astype(int))
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_tr.astype(int)]
sampler = WeightedRandomSampler(weights=torch.tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. Loss 정의
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        return torch.mean(self.alpha * (1 - pt)**self.gamma * bce_loss)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = FocalLoss(alpha=0.75, gamma=2.0)
EPOCHS = 20
patience = 4

# ==========================================
# 5. PyTorch LSTM 모델 학습 및 평가
# ==========================================
lstm_model = ChurnLSTM(vocab_size=50, embedding_dim=32, hidden_dim=64, num_layers=1).to(device)
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_loss = float("inf")
patience_counter = 0
best_lstm_path = os.path.join(MODEL_DIR, "best_lstm.pt")

print("\n" + "=" * 50)
print("🚀 [DL] PyTorch LSTM 학습 시작")
print("=" * 50)

for epoch in range(EPOCHS):
    lstm_model.train()
    train_loss = 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = lstm_model(seqs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    lstm_model.eval()
    val_loss = 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            loss = criterion(lstm_model(seqs).squeeze(), labels)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(lstm_model.state_dict(), best_lstm_path)
    else:
        patience_counter += 1
        print(f"  ⚠️ Validation Loss 개선 없음 ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"🛑 LSTM 조기 종료 발동! (Epoch {epoch+1})")
            lstm_model.load_state_dict(torch.load(best_lstm_path))
            break

# LSTM 테스트셋 예측 및 저장
lstm_model.eval()
lstm_preds_list = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        outputs = torch.sigmoid(lstm_model(seqs))
        lstm_preds_list.extend(outputs.cpu().numpy())

lstm_preds_proba = np.array(lstm_preds_list).ravel()
lstm_auc = roc_auc_score(y_test, lstm_preds_proba)
print(f"\n🚀 [LSTM 단독] Test ROC-AUC Score: {lstm_auc:.4f}")

torch.save(lstm_model.state_dict(), os.path.join(MODEL_DIR, "lstm_model_weights.pth"))
np.save(os.path.join(TEST_DIR, "lstm_preds_proba.npy"), lstm_preds_proba)
np.save(os.path.join(TEST_DIR, "test_y_true.npy"), y_test)
print("✅ LSTM 모델 및 예측값 저장 완료!")


# ==========================================
# 6. PyTorch CNN1D 모델 학습 및 평가 (추가됨)
# ==========================================
cnn_model = ChurnCNN1D(vocab_size=50, embedding_dim=32, num_filters=64).to(device)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_loss = float("inf")
patience_counter = 0
best_cnn_path = os.path.join(MODEL_DIR, "best_cnn.pt")

print("\n" + "=" * 50)
print("🚀 [DL] PyTorch CNN1D 학습 시작")
print("=" * 50)

for epoch in range(EPOCHS):
    cnn_model.train()
    train_loss = 0
    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn_model(seqs)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    cnn_model.eval()
    val_loss = 0
    with torch.no_grad():
        for seqs, labels in val_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            loss = criterion(cnn_model(seqs).squeeze(), labels)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(cnn_model.state_dict(), best_cnn_path)
    else:
        patience_counter += 1
        print(f"  ⚠️ Validation Loss 개선 없음 ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"🛑 CNN1D 조기 종료 발동! (Epoch {epoch+1})")
            cnn_model.load_state_dict(torch.load(best_cnn_path))
            break

# CNN1D 테스트셋 예측 및 저장
cnn_model.eval()
cnn_preds_list = []
with torch.no_grad():
    for seqs, _ in test_loader:
        seqs = seqs.to(device)
        outputs = torch.sigmoid(cnn_model(seqs))
        cnn_preds_list.extend(outputs.cpu().numpy())

cnn_preds_proba = np.array(cnn_preds_list).ravel()
cnn_auc = roc_auc_score(y_test, cnn_preds_proba)
print(f"\n🚀 [CNN1D 단독] Test ROC-AUC Score: {cnn_auc:.4f}")

torch.save(cnn_model.state_dict(), os.path.join(MODEL_DIR, "cnn1d_model_weights.pth"))
np.save(os.path.join(TEST_DIR, "cnn_preds_proba.npy"), cnn_preds_proba)
print("✅ CNN1D 모델 및 예측값 저장 완료!")

🚀 [DL] PyTorch 데이터 로드 시작

🚀 [DL] PyTorch LSTM 학습 시작
Epoch [1/20] | Train Loss: 0.0947 | Val Loss: 0.0859
Epoch [2/20] | Train Loss: 0.0820 | Val Loss: 0.0934
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [3/20] | Train Loss: 0.0777 | Val Loss: 0.0930
  ⚠️ Validation Loss 개선 없음 (2/4)
Epoch [4/20] | Train Loss: 0.0768 | Val Loss: 0.0774
Epoch [5/20] | Train Loss: 0.0754 | Val Loss: 0.0732
Epoch [6/20] | Train Loss: 0.0767 | Val Loss: 0.0769
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [7/20] | Train Loss: 0.0748 | Val Loss: 0.0775
  ⚠️ Validation Loss 개선 없음 (2/4)
Epoch [8/20] | Train Loss: 0.0734 | Val Loss: 0.0893
  ⚠️ Validation Loss 개선 없음 (3/4)
Epoch [9/20] | Train Loss: 0.0741 | Val Loss: 0.0782
  ⚠️ Validation Loss 개선 없음 (4/4)
🛑 LSTM 조기 종료 발동! (Epoch 9)

🚀 [LSTM 단독] Test ROC-AUC Score: 0.8545
✅ LSTM 모델 및 예측값 저장 완료!

🚀 [DL] PyTorch CNN1D 학습 시작
Epoch [1/20] | Train Loss: 0.0963 | Val Loss: 0.0846
Epoch [2/20] | Train Loss: 0.0772 | Val Loss: 0.1003
  ⚠️ Validation Loss 개선 없음 (1/4)
Epoch [3/20] | T